In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu118


In [ ]:
import os, json, random, torch, numpy as np
from datetime import datetime
from sklearn.metrics import f1_score
from datasets import Dataset, load_dataset, concatenate_dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer, pipeline
)
from huggingface_hub import login, HfApi, create_repo
from datasets import load_dataset


import os

def get_secret(label, fallback_path):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(label)
    except Exception:
        pass

    if os.path.exists(fallback_path):
        with open(fallback_path) as f:
            return json.load(f)[label]

    raise ValueError(f"Secret '{label}' not found")


HF_TOKEN = get_secret("HF_TOKEN", "/kaggle/input/secrets/secrets.json")
REPO_ID = get_secret("REPO_ID", "/kaggle/input/secrets/secrets.json")
SYNTH__DATA_REPO = get_secret("SYNTH_DATA_REPO","/kaggle/input/secrets/secrets.json")
METRICS_REPO = get_secret("METRICS_REPO")
login(token=HF_TOKEN)
COMPANY       = "MachineInnovators Inc."
COMPANY_DESC = "a leader in scalable, production-ready machine learning applications"
MODEL_REPO    = REPO_ID
BASE_MODEL    = "cardiffnlp/twitter-roberta-base-sentiment-latest"
N_PER_CLASS   = 100
LABEL_MAP     = {"negative": 0, "neutral": 1, "positive": 2}

create_repo(METRICS_REPO, repo_type="dataset", exist_ok=True)


RepoUrl('https://huggingface.co/datasets/divde/sentiment-training-metrics', endpoint='https://huggingface.co', repo_type='dataset', repo_id='divde/sentiment-training-metrics')

In [ ]:
train_data = load_dataset("tweet_eval", "sentiment", split="train")


In [ ]:
generator = pipeline(
    "text-generation",
    model="google/gemma-4-e2b-it",
    dtype=torch.float16,
    device=0,
    max_length=None,
    max_new_tokens=80,
    do_sample=True,
    temperature=1.2,
    top_p= 0.92,
    top_k=50,
    repetition_penalty=1.3
)

prompts = {
    "positive": (f"Write a short positive customer review or social media post about "
                 f"{COMPANY}, {COMPANY_DESC}. Output only the text.No additional comments or introduction."
                 f"here some stylistic examples with various sentiments{random.choices(train_data['text'], k = 5)}"),

    "negative": (f"Write a short negative complaint or post about "
                 f"{COMPANY}, {COMPANY_DESC}. Output only the text.No additional comments or introduction."
                 f"here some stylistic examples with various sentiments{random.choices(train_data['text'], k = 5)}"),

    "neutral":  (f"Write a short neutral news update or factual statement about" 
                 f"{COMPANY}, {COMPANY_DESC}. Output only the text.No additional comments or introduction."
                 f"here some stylistic examples with various sentiments{random.choices(train_data['text'], k = 5)}"),
}

from tqdm.auto import tqdm

def generate_examples(label, n, batch_size=20):
    examples = []
    prompt = prompts[label]
    with tqdm(total=n, desc=label) as pbar:
        for _ in range(0, n, batch_size):
            current_batch = min(batch_size, n - len(examples))
            outputs = generator(
                [[{"role": "user", "content": prompt}]] * current_batch,
            )
            for out in outputs:
                text = out[0]["generated_text"][-1]["content"].strip()
                if text:
                    examples.append({"text": text, "label": LABEL_MAP[label]})
            pbar.update(current_batch)
    return examples

data = []
for label in LABEL_MAP:
    print(f"Generating {N_PER_CLASS} {label}...")
    data.extend(generate_examples(label, N_PER_CLASS))

random.shuffle(data)
print(f"Total: {len(data)}")

print(random.choices(data, k=10))


del generator
torch.cuda.empty_cache()


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Generating 300 negative...


negative:   0%|          | 0/300 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
real_test = load_dataset("tweet_eval", "sentiment", split="test")
real_test = real_test.map(tokenize, batched=True)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, add_prefix_space=True)

def tokenize(batch):
    return tokenizer(batch["text"], max_length=128, truncation=True)

split = Dataset.from_list(data).train_test_split(test_size=0.15, seed=42)

combined = 

train_ds = split["train"].map(tokenize, batched=True)
test_ds  = split["test"].map(tokenize, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Train: {len(train_ds)} | Test: {len(test_ds)}")


In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {"f1": f1_score(labels, preds, average="weighted")}

current = AutoModelForSequenceClassification.from_pretrained(MODEL_REPO)
baseline_metrics = Trainer(
    model=current,
    eval_dataset=real_test,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    data_collator=collator,
).evaluate()
baseline_f1 = baseline_metrics["eval_f1"]
print(f"Baseline F1: {baseline_f1:.4f}")

del current
torch.cuda.empty_cache()


In [ ]:
new_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=3)

trainer = Trainer(
    model=new_model,
    args=TrainingArguments(
        output_dir="./results",
        hub_model_id=MODEL_REPO,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
    ),
    train_dataset=train_ds,
    eval_dataset=real_test,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    data_collator=collator,
)

trainer.train()
new_metrics = trainer.evaluate(eval_dataset= real_test)
new_f1 = new_metrics["eval_f1"]
print(f"New F1: {new_f1:.4f}")


In [ ]:
quality_control = new_f1 >= 0.8
print(f"Improved: {quality_control}  ({baseline_f1:.4f} → {new_f1:.4f})")

payload = {
    "timestamp":  datetime.now().isoformat(),
    "baseline_f1": round(baseline_f1, 4),
    "new_f1":      round(new_f1, 4),
    "performance": quality_control,
    "company":     COMPANY,
    "n_samples":   len(data),
}
try:
    path = hf_hub_download(repo_id=METRICS_REPO, filename="history.json", repo_type="dataset")
    history = json.load(open(path))
except:
    history = []

history.append(current_metrics)

api = HfApi(token=HF_TOKEN)

if quality_control:
    trainer.push_to_hub(
        commit_message=f"retrain: F1 {baseline_f1:.3f}→{new_f1:.3f}"
    )


api.upload_file(
    path_or_fileobj=json.dumps(history).encode(),
    path_in_repo="history.json",
    repo_id=METRICS_REPO,
    repo_type="dataset",
    commit_message=f"run {datetime.now().strftime('%Y-%m-%d %H:%M')}"
)

api.create_tag(
    repo_id=REPO_ID,
    tag=f"v{payload["timestamp"]}",
    message=f"F1={new_f1:.3f}"
)

try:
    existing_dataset = load_dataset(SYNTH__DATA_REPO, split="train")
except:
    create_repo(SYNTH__DATA_REPO, repo_type="dataset")

dataset_dict = DatasetDict({
    "train": train_ds,
    "test": test_ds
})

dataset_dict.push_to_hub(SYNTH__DATA_REPO)


: 